<h3><b>1. 라이브러리 가져오기 

필요 라이브러리 설치하기 

In [ ]:
!pip install python-barcode

라이브러리 가져오기

In [ ]:
import barcode
import cv2

바코드 패키지에서 사용가능한 것 확인하기

In [ ]:
dir(barcode)

<h3><b>2. ISBN-13 바코드 생성하기 

ISBN-13은 도서 관련 국제 순서 표기 번호이다. 

ISBN-13 관련 라이브러리 가져오기 및 ImageWriter 가져오기 

In [ ]:
from barcode import ISBN13
from barcode.writer import ImageWriter
import cv2

In [ ]:
ImageWriter()

number를 문자열로 정의하기

In [ ]:
num = '97910456752421'
num

ISBN-13 클래스 초기화 하기 

In [ ]:
book_barcode = ISBN13(num, writer=ImageWriter())
book_barcode

파일 생성 및 저장 위치 설정하기 

In [ ]:
import os


outputPath = 'D:\\Develops\\barcode\\'
os.makedirs(outputPath, exist_ok=True)

In [ ]:
output = outputPath + 'isbn13_barcode1'
output

바코드 이미지 저장 및 확인하기 

In [ ]:
book_barcode.save(output)

In [ ]:
import numpy as np

full_image_path = output + '.png'

# 1. Read the file into a numpy byte array first
img_array = np.fromfile(full_image_path, np.uint8)

# 2. Decode the byte array into an OpenCV image
a = cv2.imdecode(img_array, cv2.IMREAD_COLOR)

# 3. Display the image
cv2.imshow('barcode', a)
cv2.waitKey(0)
cv2.destroyAllWindows()

<h3><b>3. 카메라로 바코드 인식 바운딩 박스 생성하기 

카메라 불러오기

In [ ]:
#!pip install pyzbar

In [1]:
import cv2
from pyzbar.pyzbar import decode

바코드 인식 함수 생성하기 

In [ ]:
def barcodeReader(frame):
    # 1. 흑백(Grayscale) 이미지로 변환
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # 2. Otsu 알고리즘을 이용한 이진화 (Thresholding)
    # 웹캠의 어두운/밝은 영역을 계산해 첨부해주신 정제된 PNG 이미지처럼 완벽한 흑백 상태로 만듭니다.
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)

    # 3. 이진화된 깔끔한 이미지(thresh)를 기반으로 바코드 인식 시도
    detected_barcodes = decode(thresh)

    # 만약 환경에 따라 이진화로 못 찾은 경우, 원본 그레이스케일로 2차 인식(보완책)
    if not detected_barcodes:
        detected_barcodes = decode(gray)

    # 바코드 식별을 위한 최소 크기 설정 
    min_width = 10 
    min_height = 10 
    
    if detected_barcodes:
        for barcode in detected_barcodes:
            (x, y, w, h) = barcode.rect
            
            # 노이즈 오인식 방지를 위해 최소 임계값 이상의 바코드만 유효 처리
            if w > min_width and h > 10:
                # 인식된 바코드에 초록색 바운딩 박스 생성 (결과값이 잘 보이게 간격 약간 조절)
                cv2.rectangle(frame, (x - 5, y - 5), (x + w + 5, y + h + 5), (0, 255, 0), 3)
                
                if barcode.data:
                    # 바코드 데이터 디코딩 (예: "9791045675247")
                    barcode_data = barcode.data.decode("utf-8")
                    barcode_type = barcode.type
                    
                    # 4. 웹캠 화면에 결과값 텍스트 출력
                    text = f"{barcode_data} ({barcode_type})"
                    cv2.putText(frame, text, (x, y - 15), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2, cv2.LINE_AA)
                    
                    # 콘솔(터미널)에 결과값 출력
                    print(f"[인식 완료] 결과값: {barcode_data} | 타입: {barcode_type}")

    return frame

결과 확인하기 

In [3]:
if __name__ == "__main__":
    # 카메라 작동시키기 
    cap = cv2.VideoCapture(0)
    
    # 3. 카메라 해상도 높이기 (HD 또는 FHD 권장)
    # 픽셀이 뭉개지지 않아야 바코드의 얇은 선들을 정확히 구분할 수 있습니다.
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

    # 오토포커스 활성화 시도 (지원하는 웹캠인 경우)
    cap.set(cv2.CAP_PROP_AUTOFOCUS, 1)

    while True:
        ret, frame = cap.read()

        if ret:
            barcodeReader(frame)

        # 'q'를 누르면 종료
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

<h3><b>4. 바코드 인식 결과 값에 따른 MC-270 움직임 제어하기 

In [ ]:
import cv2
from pyzbar.pyzbar import decode
from pymycobot.mecharm import MechArm
import time

In [ ]:
mc = MechArm("com3", 115200)

바코드 인식 후 MC-270 움직임 제어 함수 

In [ ]:
def move_mechArm(barcode_data):
    if barcode_data == "9784567524216":
        print("Moving!")
        mc.send_angles([28.3, 55.98, -8.43, -1.23, 20.83, 1.58], 30)
        time.sleep(2)
        mc.send_angles([-33.66, -28.38, -8.43, 0.35, 37.88, 1.58], 30)
        time.sleep(2)
        mc.send_angles([0, 0, 0, 0, 0, 0], 30)
        
    else:
        print("Not Found the barcode")

바코드 인식 함수 

In [ ]:
def barcodeReader(frame):
    # 입력된 프레임에서 바코드 찾기 
    detected_barcodes = decode(frame)

    # 바코드 식별을 위한 최소 크기 설정 
    min_width = 10  # 바코드 영역의 최소 너비 임계값 
    min_height = 10  # 바코드 영역의 최소 높이 임계값 
    
    if not detected_barcodes:
        print("Barcode Not Detected or your barcode is blank/corrupted!")
    else:
        # 바코드 인식 시, 바코드의 위치와 크기 가져오기 
        for barcode in detected_barcodes:
            (x, y, w, h) = barcode.rect
            
            # 바코드 너비와 높이가 최소 임계값보다 큰지 확인 >> 유효한 바코드인지 확인하기 
            if w > min_width and h > min_height:
                # 바코드에 바운딩 박스 생성 
                cv2.rectangle(frame, (x-10, y-10), (x + w+10, y + h+10), (255, 0, 0), 2)
                if barcode.data:
                    barcode_data = barcode.data.decode("utf-8")
                    barcode_type = barcode.type
                    print("Barcode Data:", barcode_data)
                    print("Barcode Type:", barcode_type)
                    move_mechArm(barcode_data)
                    # time.sleep(6)

    cv2.imshow("Barcode Detection", frame)

In [ ]:
if __name__ == "__main__":
    cap = cv2.VideoCapture(0)

    while True:
        ret, frame = cap.read()

        if ret:
            barcodeReader(frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()